# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates the loading, exploration, and analysis of the dataset **"Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields (columns).

In [ ]:
# List all record sets by @id and their fields by @id
record_sets = metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()

# Save first record set for demonstration
if len(record_sets) > 0:
    main_record_set = record_sets[0]
    main_record_set_id = main_record_set.id
    main_field_ids = [f.id for f in main_record_set.fields]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we extract all record sets and load them as DataFrames using their `@id`. For demonstration, the main record set identified above is used.

In [ ]:
# Extract data from each record set
dataframes = {}
all_record_set_ids = [rs.id for rs in record_sets]

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Below, we select a numeric field, filter for values above a threshold, normalize it, and optionally group by a categorical field.

In [ ]:
# Choose a likely numeric field and group field based on typical proteomics columns.
numeric_field = None
possible_numeric_fields = [
    'cr:protein_coverage_percent',
    'cr:peptide_count',
    'cr:molecular_weight',
    'cr:normalized_abundance',
    'cr:total_abundance',
    'cr:unique_peptide_count'
]
for f in possible_numeric_fields:
    if f in dataframes[main_record_set_id].columns:
        numeric_field = f
        break

if numeric_field is not None:
    threshold = dataframes[main_record_set_id][numeric_field].dropna().mean()
    
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by a potential categorical field, such as 'cr:sample' or 'cr:accession'
    group_fields = ['cr:sample_id', 'cr:accession', 'cr:gene', 'cr:protein_family']
    group_field = None
    for gf in group_fields:
        if gf in filtered_df.columns:
            group_field = gf
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("Could not find a numeric field for analysis among expected names.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field, if available
if numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If normalized, plot normalized values histogram as well
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True)
        plt.title(f'Normalized Distribution of {numeric_field}')
        plt.xlabel(norm_col)
        plt.ylabel('Count')
        plt.show()

## 6. Conclusion
In this notebook, we loaded a structured proteomics dataset as described in its FAIR Croissant schema. Using `mlcroissant`, we explored available record sets and fields, extracted tabular data, performed filtering and normalization on numeric columns, and visualized basic distributions. Such a workflow facilitates transparent, reproducible data science directly from machine-readable dataset metadata.